# Scoring and audit, part 1

Stage 1 (Cells 0 to 7) scores the six stage-main caches with the gate-E scorer and reads what
needs no human. Stage 2 (Cells 8 to 14) runs the statistics reader, HT-4 and Gate F
(DESIGN_ADDENDUM O, D270 to D278). Script: `scripts/scoring_audit.py`.

**Runtime: CPU.** No API call, no GPU, no cost. Run the cells in order. Every cell prints ids
and counts only. Files that hold answer text are named `TEXT_*.csv` and stay in Drive.

People: one runner. The runner reads the outputs, so the runner is never an HT-4 checker.
Nobody who may become an HT-4 checker opens this notebook's outputs or Drive `scored/`
(D168).

Before Cell 0: `scripts/scoring_audit.py` and this notebook are committed on `main` of
`banglaindoorvqa-main` (Cell 0 line C06 fails otherwise). Commit the notebook with its outputs
cleared.

Send back in the chat the printouts each cell asks for. Before Cell 13 or 14 runs on a new
runtime, run Cells P and 0 again (they define `run`, `sh` and the token).

## Cell P: settings
Set `REPO_SLUG` to the owner of the repo `banglaindoorvqa-main` (not the dataset repo
`banglaindoorvqa`). Nothing else changes.

In [ ]:
# Cell P: settings
DRIVE_ROOT = "/content/drive/MyDrive/BanglaIndoorVQA"   # the Drive project folder
REPO_SLUG = "OWNER/banglaindoorvqa-main"                 # set OWNER
MAIN = "/content/banglaindoorvqa-main"                   # the main clone
GATE_E = "/content/bivqa_gateE"                          # detached checkout of the tag gate-E
SCRIPT = MAIN + "/scripts/scoring_audit.py"
TAGGER_NAME = ""     # Cell 14 only: the name on the annotated tag gate-F
TAGGER_EMAIL = ""    # Cell 14 only: the e-mail on the tag (for example your GitHub no-reply address)
# The reported deviation of HT-4 round 1 (D279); used by Cells 13 and 14. Do not edit.
DEVIATION = ("D279: HT-4 round 1 scorer = consensus on 187 of 200 (93.5%), below the pre-set 95%; "
             "scorer 0.2.2-stage2 kept as primary; sensitivity rules a and b re-scored")

## Cell 0: clone, pull, gate-E checkout, input check
Mounts Drive, clones `banglaindoorvqa-main` with the Colab secret `GITHUB_TOKEN` (the token is
never printed or written to disk), pulls, makes the detached gate-E checkout, then runs the
read-only check.

**Correct output:** the Git steps end with `SETUP: PASS`, then lines C01 to C30, each `PASS`
or `INFO`, then `CHECK: ALL PASS (30 lines)`. C05 should read `tests/: same as gate-E`.
C27 lists the kept harness fields. A line `NOTE: the pilot scoring kept fields that these
caches lack` is not a fail, but send it in the chat. Any `FAIL`: stop and send the whole
printout.

In [ ]:
# Cell 0: setup and the input check
import os, subprocess, sys
from google.colab import drive, userdata

os.environ["PYTHONDONTWRITEBYTECODE"] = "1"      # no __pycache__ in the gate-E checkout (D252)
sys.dont_write_bytecode = True
drive.mount("/content/drive")

if "OWNER" in REPO_SLUG:
    raise SystemExit("FAIL: set REPO_SLUG in Cell P")
if not REPO_SLUG.endswith("/banglaindoorvqa-main"):
    raise SystemExit("FAIL: REPO_SLUG must name banglaindoorvqa-main, not the dataset repo")
try:
    TOKEN = userdata.get("GITHUB_TOKEN")
except Exception as exc:
    raise SystemExit("FAIL: the Colab secret GITHUB_TOKEN is missing or not allowed for this "
                     "notebook (%s)" % type(exc).__name__)
if not TOKEN:
    raise SystemExit("FAIL: the Colab secret GITHUB_TOKEN is empty")
PLAIN = "https://github.com/%s.git" % REPO_SLUG
AUTHED = "https://x-access-token:%s@github.com/%s.git" % (TOKEN, REPO_SLUG)


def sh(*args):
    p = subprocess.run(list(args), capture_output=True, text=True)
    shown = " ".join(args).replace(TOKEN, "***")
    out = (p.stdout + p.stderr).replace(TOKEN, "***").strip()
    print("%s  %s%s" % ("PASS" if p.returncode == 0 else "FAIL", shown,
                        ("\n" + out) if out else ""))
    if p.returncode:
        raise SystemExit("SETUP: FAIL (exit %d)" % p.returncode)
    return out


if not os.path.isdir(MAIN + "/.git"):
    sh("git", "clone", "-q", AUTHED, MAIN)
sh("git", "-C", MAIN, "remote", "set-url", "origin", PLAIN)       # no token in .git/config
sh("git", "-C", MAIN, "checkout", "-q", "main")
sh("git", "-C", MAIN, "fetch", "-q", AUTHED,
   "+refs/heads/main:refs/remotes/origin/main", "+refs/tags/*:refs/tags/*")
sh("git", "-C", MAIN, "merge", "-q", "--ff-only", "origin/main")   # the pull
sh("git", "-C", MAIN, "worktree", "prune")
if not os.path.isdir(GATE_E):
    sh("git", "-C", MAIN, "worktree", "add", "-q", "--detach", GATE_E, "gate-E")
print("main HEAD", sh("git", "-C", MAIN, "rev-parse", "--short=12", "HEAD").splitlines()[-1])
print("gate-E checkout HEAD", sh("git", "-C", GATE_E, "rev-parse", "--short=12", "HEAD").splitlines()[-1])
print("SETUP: PASS")


def run(*args):
    cmd = [sys.executable, SCRIPT, "--root", DRIVE_ROOT, "--gate-e", GATE_E, "--main", MAIN] + list(args)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         env=dict(os.environ, PYTHONDONTWRITEBYTECODE="1"))
    for line in p.stdout:
        print(line, end="")
    rc = p.wait()
    print("exit code %d" % rc)
    return rc


run("check")

## Cell 1: selftest on made-up answers
Builds a small made-up project in a temporary folder and runs the whole chain with the real
gate-E scorer. Reads no project data.

**Correct output:** `SELFTEST: 47 of 47 checks PASS`.

In [ ]:
# Cell 1: selftest
run("selftest")

## Cell 2: score the six caches
One gate-E `asset9_scorer.py score` command per cache, run from the gate-E checkout (D270,
D271). Each is checked against its cache straight after. A scored file that exists is skipped;
`--overwrite` replaces it, but never once an HT-4 sheet exists.

**Correct output:** for each model the scorer's own printout, then a `PASS` line:

| model | records | scored | excluded |
|---|---|---|---|
| qwen | 15,947 | 15,790 | 157 |
| gemma | 11,600 | 11,485 | 115 |
| gemini | 11,600 | 11,485 | 115 |
| qwen_ft_s42, s43, s44 | 4,347 each | 4,305 each | 42 each |

(Excluded: 9 per P-CORE-CLOSED condition, 9 per P-NAME and P-CLUST condition, 8 per P-FCE
condition, 14 per R-TEST condition.)

qwen also prints `free check: 738 of 738 copies score exactly as their EXP-L source (6 of them
excluded items), 0 differ`. The cell ends with `SCORE: qwen PASS, gemma PASS, ...` for all six.
`Skipped` must not appear in any scorer printout.

In [ ]:
# Cell 2: score (one model at a time is fine: run("score", "--model", "gemma"))
run("score")

## Cell 3: verify
Reads the six scored files again and checks them against their caches and manifests; repeats
the free check of the 738 copies (addendum L10). Read-only.

**Correct output:** six `PASS` lines with the counts of Cell 2, the free-check line, then
`All six files: 52188 records, 51675 scored, 513 excluded` and `VERIFY: ALL PASS`.

In [ ]:
# Cell 3: verify
run("verify")

## Cell 4: D192, D194 and open-issue counts (D272)
Counts only. The answers behind the counts go to `scored/main/reading/TEXT_*.csv` for HT-4
and HT-5; do not paste those files in the chat.

**Correct output:** a D192 table with seven findings (d192_5 and d192_6 split by gold), a
D194 table per model, the forced-choice open issues per model and condition, and `COUNTS: DONE`.
These are results; there is no expected value.

In [ ]:
# Cell 4: counts
run("counts")

## Cell 5: D159 against main (D273)
The 97 D5 pairs (L-BB 58, N-BB 39): the pilot against main (the D159 reading), and the D5
repeat and the Batch test against main beside it (information).

**Correct output:** three blocks, each `pairs 97 (L-BB 58, N-BB 39)`, and `D159: DONE`.

In [ ]:
# Cell 5: D159
run("d159")

## Cell 6: the 10-item check
Ten seeded-random scored answers: 4 open with the image, 2 without, 4 forced choice (one per
forced-choice condition), the make-up of D166.

**Correct output:** ten lines S01 to S10 (model, condition, qa_id) and the path of
`TEXT_sample10.csv`. Send that file in the chat.

In [ ]:
# Cell 6: sample10
run("sample10")

## Cell 7: status
What exists so far. Read-only.

In [ ]:
# Cell 7: status
run("status")

# Stage 2: HT-4 and Gate F

People (D275): the runner prepares the sheets and is never a checker. Two checkers, different
people, neither of them the runner. They never see this notebook's outputs, and they do not open
Drive `scored/` or `human/HT-4/key/` until both sheets are in (D168).

## Cell 8: the statistics reader on the scored files
Runs `stats.py` `build_world` and `load_scored` (gate-E copy) on the six scored files, as the
Statistics chat will. Read-only.

**Correct output:** about 20 lines that start with `PASS` (no `FAIL`), then `LOADTEST: PASS`.

In [ ]:
# Cell 8: loadtest
run("loadtest")

## Cell 9: draw the HT-4 sheets
Draws the 200-answer sample and the 40 open-issue answers beside it (D274, D276), and writes two
sheets with their own row order and codes, the written rules and the key. Run it once; a second
run changes nothing.

**Correct output:** `Pools found: phrase_correct 10, multi_cued 18, multi_uncued 187,
fc_candidates 28, named 2`; `Sample: 200 answers`; every scorer label with at least 10 where the
frame has 10; `Rows per checker sheet:` about 240; `HT-4 DRAW: DONE`.

Then send `human/HT-4/to_send/HT4_sheet_C1.xlsx` to checker 1 and `HT4_sheet_C2.xlsx` to checker 2.
Send nothing else, and never the `key/` folder.

In [ ]:
# Cell 9: ht4-draw
run("ht4-draw")

## Cell 10: receive the returned sheets
Save the returned files, unchanged, as `human/HT-4/received/HT4_sheet_C1_returned.xlsx` and
`HT4_sheet_C2_returned.xlsx`, then run this cell. It may run as often as needed; a receipt
never changes.

**Correct output:** `C1: PASS ...`, `C2: PASS ...`, `RECEIPT: 2 of 2`, the agreement counts, and
the path of `to_send/HT4_consensus.xlsx` (or `No row differs`). Give the consensus sheet to the
two checkers together.

In [ ]:
# Cell 10: ht4-receive
run("ht4-receive")

## Cell 11: consensus and the D238 file
Save the filled consensus sheet as `human/HT-4/received/HT4_consensus_returned.xlsx` (skip this if
no row differed), then run this cell.

**Correct output:** `Consensus sheet: n rows settled`, then the three files written, then
`CONSENSUS: DONE`.

In [ ]:
# Cell 11: ht4-consensus
run("ht4-consensus")

## Cell 12: the HT-4 reading
Prints lines H01 to H13 and HT-4: PASS or FAIL. Round 1 was read on 2026-10-23: FAIL, 187 of 200
(D279). Do not run it again unless asked; a second run only rewrites the same reading.

In [ ]:
# Cell 12: ht4-read
run("ht4-read")

## Cell 12b: D279 sensitivity re-score
Re-scores every answer in memory with the two sensitivity rules (a: of names next to each other only
the last counts; b: a refusal phrase plus two or more letters is invalid). The scored files and the
scorer do not change. Writes ids and labels only to `scored/main/sensitivity_D279/`.

**Correct output:** `Answers changed:` a few hundred for rule a (most change only the label, not
the score) and 10 for rule b; a table per model and condition, the change of P1, P2, P12 and P3 in points, then `SENS: DONE`. Send the whole printout.

In [ ]:
# Cell 12b: sens
run("sens")

## Cell 13: Gate F checks (no tag)
Runs every Gate F check and the 445 unit tests, with the reported deviation D279, and tags nothing.

**Correct output:** F01 `PASS ... reported deviation: D279 ...; D279 sensitivity re-score done on
these scored files`, F02 to F08 `PASS` (F07: `445 run, result 'OK'`), then
`GATE F: ALL CHECKS PASS (check only; nothing tagged)`.

In [ ]:
# Cell 13: gate-f --check-only
run("gate-f", "--check-only", "--deviation", DEVIATION)

## Cell 14: the tag gate-F
Only after Cell 13 passed. Fill `TAGGER_NAME` and `TAGGER_EMAIL` in Cell P first. Creates the
annotated tag on main's HEAD and pushes it.

**Correct output:** F01 to F08 `PASS`, `PASS tag gate-F: tag, on <commit>`, then the push lines
and `TAG gate-F: PUSHED (<sha>)`.

In [ ]:
# Cell 14: tag and push gate-F
if not TAGGER_NAME or not TAGGER_EMAIL:
    raise SystemExit("FAIL: fill TAGGER_NAME and TAGGER_EMAIL in Cell P")
rc = run("gate-f", "--tagger-name", TAGGER_NAME, "--tagger-email", TAGGER_EMAIL,
         "--deviation", DEVIATION)
if rc == 0:
    sh("git", "-C", MAIN, "push", "-q", AUTHED, "refs/tags/gate-F")
    local = sh("git", "-C", MAIN, "rev-parse", "refs/tags/gate-F").splitlines()[-1]
    remote = sh("git", "-C", MAIN, "ls-remote", AUTHED, "refs/tags/gate-F").split()
    ok = bool(remote) and remote[0] == local
    print("TAG gate-F: %s (%s)" % ("PUSHED" if ok else "NOT ON GITHUB", local[:12]))